# Modelo ML XGBOOST CLASSIFIER

In [4]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import joblib
import json
# ========================
# A. Cargar datos
# ========================
X_train = pd.read_csv("../models/x_train_sel.csv")
y_train = pd.read_excel("../data/processed/X&Ys/y_train.xlsx").squeeze()
# ========================
# B. Cargar mapping sin modificarlo
# ========================
with open("../data/processed/Json/ciudad_transformation_rules.json") as f:
    ciudad_mapping = json.load(f)  # formato: {nombre: id}
# Invertir el mapping para usarlo
id_to_ciudad = {str(v): k for k, v in ciudad_mapping.items()}
y_train_nombres = y_train.astype(str).map(id_to_ciudad)
# ========================
# C. Codificar nombres
# ========================
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_nombres)
# ========================
# D. Asegurarse de que las columnas numéricas estén incluidas
# ========================
columnas_extra = [
    "estimated_price_eur_x",
    "estimated_price_eur_y",
    "distance_to_city_center_km",
    "class_n"
]
columnas_finales = columnas_extra + [
    col for col in X_train.columns
    if col not in columnas_extra  # Evita duplicados
]
X_train_final = X_train[columnas_finales]
# ========================
# E. Entrenar modelo
# ========================
model = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
model.fit(X_train_final, y_train_encoded)
# ========================
# F. Evaluar
# ========================
y_pred = model.predict(X_train_final)
print(":marca_de_verificación_gruesa: Accuracy:", accuracy_score(y_train_encoded, y_pred))
# ========================
# G. Guardar modelo y encoder
# ========================
joblib.dump(model, "../models/model_completo.pkl")
joblib.dump(le, "../models/label_encoder.pkl")
pd.Series(X_train_final.columns).to_csv("../data/processed/x_train_columns.csv", index=False, header=False)

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import json


train_data = pd.read_csv("../models/x_train_sel.csv")
test_data = pd.read_csv("../models/x_test_sel.csv")

X_train = train_data.drop(["ciudad"], axis=1)
y_train = train_data["ciudad"]
X_test = test_data.drop(["ciudad"], axis=1)
y_test = test_data["ciudad"]

train_data.head()


ModuleNotFoundError: No module named 'xgboost'

In [ ]:

# 4. Entrenar modelo
model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
model.fit(X_train, y_train)

# 5. Evaluar
y_pred = model.predict(X_test)
print("✔️ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Clasificación:\n", classification_report(y_test, y_pred))

# 🧪 PREDICCIÓN PERSONALIZADA

In [ ]:
X_train.columns.to_frame().to_csv("../data/processed/x_train_columns.csv", index=False, header=False)

# PASO 2: Definir función para construir entrada de usuario

import pandas as pd

def construir_input_usuario(valores_activados, columnas_referencia_path):
    columnas_modelo = pd.read_csv(columnas_referencia_path, header=None).squeeze().tolist()
    input_df = pd.DataFrame(columns=columnas_modelo)
    input_df.loc[0] = 0
    for col, val in valores_activados.items():
        if col in input_df.columns:
            input_df.at[0, col] = val
    return input_df

# PASO 3: Simular selección del usuario
seleccion_usuario = {
    "perfil_Familia": 1,
    "entorno_Ciudad": 1,
    "Clasificacion_Relax": 1,
    "temporada_Verano": 1,
    "origin_city_París": 1,
    "flight_price": 120,
    "estimated_price_eur_y": 90,
    "temp_max": 30,
    "temp_min": 20
}

X_user = construir_input_usuario(
    seleccion_usuario,
    columnas_referencia_path="../data/processed/x_train_columns.csv"
)

with open("../data/processed/Json/ciudad_transformation_rules.json", "r") as f:
    ciudad_mapping = json.load(f)

id_to_ciudad = {str(v): k for k, v in ciudad_mapping.items()}

y_train_original = pd.read_excel("../data/processed/X&Ys/y_train.xlsx").squeeze()
y_train_original = y_train_original.astype(str).map(id_to_ciudad)

le = LabelEncoder()
le.fit(y_train_original)


In [ ]:
probs = model.predict_proba(X_user)[0]
top5_indices = np.argsort(probs)[::-1][:5]
top5_labels = model.classes_[top5_indices]
top5_ciudades = le.inverse_transform(top5_labels)

print("🏝️ Top 5 ciudades recomendadas:", top5_ciudades)

# 🌍 FUNCIONES DE ENRIQUECIMIENTO (usando df original)

In [ ]:
full_df = pd.read_csv("../data/processed/total_data_240k.csv")

temporada_usuario = "Verano" 

def get_clima_estimado(ciudad, temporada):
    clima = full_df[
        (full_df['ciudad'] == ciudad) &
        (full_df['temporada'].str.strip().str.lower() == temporada.strip().lower())
    ]
    if clima.empty:
        return ""
    datos = clima[['temp_max', 'temp_min', 'precipitacion']].mean().round(1).to_dict()
    return f"{datos}"


def get_eventos(ciudad, temporada):
    eventos = full_df[
        (full_df['ciudad'] == ciudad) &
        (full_df['temporada'].str.strip().str.lower() == temporada.strip().lower())
    ][['evento_nombre', 'evento_categoria', 'evento_desc', 'fecha']]
    eventos = eventos.dropna().drop_duplicates().head(3)
    if eventos.empty:
        return ""
    return "\n".join(
        f"- {row['evento_nombre']} ({row['evento_categoria']}): {row['evento_desc']} [{row['fecha']}]"
        for _, row in eventos.iterrows()
    )

def get_precio_vuelo(origen, destino):
    vuelos = full_df[(full_df['origin_city'] == origen) & (full_df['ciudad'] == destino)]
    return round(vuelos['flight_price'].mean(), 2) if not vuelos.empty else "Sin datos"

def get_hotel(ciudad):
    hoteles = full_df[full_df['ciudad'] == ciudad][['hotel_name', 'estimated_price_eur_y', 'hotel_type', 'distance_to_city_center_km']]
    hotel = hoteles.dropna().sort_values(by='estimated_price_eur_y').head(1)
    return hotel.to_dict(orient='records')[0] if not hotel.empty else "Sin hoteles"


# 📦 MOSTRAR INFO ENRIQUECIDA

In [ ]:
for ciudad in top5_ciudades:
    print(f"\n🌍 Ciudad: {ciudad}")
    clima = get_clima_estimado(ciudad, temporada_usuario)
    if clima:
        print("☁️ Clima estimado:", clima)

    eventos = get_eventos(ciudad, temporada_usuario)
    if eventos:
        print("🎫 Eventos:\n", eventos)

    print("✈️ Vuelo desde origen:", get_precio_vuelo("Madrid", ciudad))
    print("🏨 Hotel recomendado:", get_hotel(ciudad))